# 📥 01_data_ingestion

Processo de recolha e carregamento dos dados utilizados no projeto.

Inclui:
 Import libraries
 Define project paths
 Download market data (S&P500)
 Save raw market dataset
 Download financial news
 Save raw news dataset

- CELL 1 — IMPORT LIBRARIES
- CELL 2 — DEFINE PROJECT PATHS
- CELL 3 — DEFINE FINANCIAL INSTRUMENT
- CELL 4 — DOWNLOAD HISTORICAL MARKET DATA
- CELL 5 — CLEAN DATAFRAME STRUCTURE
- CELL 6 — STANDARDIZE COLUMN TYPES
- CELL 7 — ADD TICKER COLUMN
- CELL 8 — REORDER DATASET COLUMNS
- CELL 9 — DATASET INSPECTION
- CELL 10 — SAVE RAW DATASET

In [1]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ------------------------------------------------------------
# Objective:
# Import the Python libraries required for the market data
# ingestion stage of the research pipeline.
#
# Libraries used:
#
# - pandas: data manipulation
# - yfinance: financial data API
# - pathlib: path management
# ============================================================

import pandas as pd
import yfinance as yf
from pathlib import Path            # HTTP requests

In [2]:
# ============================================================
# CELL 2 — DEFINE PROJECT PATHS
# ------------------------------------------------------------
# Objective:
# Define the folder structure used to store datasets.
#
# Raw datasets will be stored in:
#
# ML_STOCKS_LSTM/datasets/raw
# ============================================================

from pathlib import Path

# go to project root (ML_STOCKS_LSTM)
project_root = Path.cwd().parent

# datasets folder
datasets_dir = project_root / "datasets"

# raw data folder
raw_dir = datasets_dir / "raw"

# create folder if needed
raw_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data directory:", raw_dir)

Project root: c:\Users\anton\0______MASTER_IN_DATA_SCIENCE\1_SEMESTRE\RESEARCH_METHODOLOGIES\RESEARCH_METHODOLOGIES_2\ML_Stocks_LSTM
Raw data directory: c:\Users\anton\0______MASTER_IN_DATA_SCIENCE\1_SEMESTRE\RESEARCH_METHODOLOGIES\RESEARCH_METHODOLOGIES_2\ML_Stocks_LSTM\datasets\raw


In [3]:
# ============================================================
# CELL 3 — DEFINE FINANCIAL INSTRUMENT
# ------------------------------------------------------------
# Objective:
# Define the financial asset that will be downloaded.
#
# For this project we use the S&P 500 index as a proxy for
# the overall US equity market.
#
# Yahoo Finance ticker:
#
# ^GSPC
# ============================================================

ticker = "^GSPC"

print("Selected ticker:", ticker)

Selected ticker: ^GSPC


In [4]:
# ============================================================
# CELL 4 — DOWNLOAD HISTORICAL MARKET DATA
# ------------------------------------------------------------
# Objective:
# Retrieve daily historical price data from Yahoo Finance.
#
# auto_adjust=True ensures prices are adjusted for:
#
# - stock splits
# - dividends
#
# This produces a consistent historical price series.
# ============================================================

sp500 = yf.download(
    ticker,
    start="2000-01-01",
    auto_adjust=True,
    progress=False
)

print("Dataset downloaded successfully.")
print("Number of rows:", len(sp500))

Dataset downloaded successfully.
Number of rows: 6612


In [5]:
# ============================================================
# CELL 5 — CLEAN DATAFRAME STRUCTURE
# ------------------------------------------------------------
# Objective:
# Convert the dataset into a clean dataframe format.
#
# Actions performed:
#
# - convert index to column
# - flatten multi-index columns if necessary
# - remove column index name
# ============================================================

# convert index to column
sp500 = sp500.reset_index()

# flatten multi-index columns if necessary
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

# remove column index name
sp500.columns.name = None

print("Columns in dataset:")
print(sp500.columns)

Columns in dataset:
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')


In [6]:
# ============================================================
# CELL 6 — STANDARDIZE COLUMN TYPES
# ------------------------------------------------------------
# Objective:
# Ensure correct data types for the dataset.
#
# Actions performed:
#
# - convert Date column to datetime
# - ensure chronological ordering
# ============================================================

sp500["Date"] = pd.to_datetime(sp500["Date"])

# sort dataset chronologically
sp500 = sp500.sort_values("Date")

print("Date column type:", sp500["Date"].dtype)

Date column type: datetime64[s]


In [7]:
# ============================================================
# CELL 7 — ADD TICKER COLUMN
# ------------------------------------------------------------
# Objective:
# Add a ticker identifier column.
#
# Even though the current dataset contains only one asset,
# including the ticker column ensures compatibility with
# future multi-asset extensions.
# ============================================================

sp500["ticker"] = ticker

In [8]:
# ============================================================
# CELL 8 — REORDER DATASET COLUMNS
# ------------------------------------------------------------
# Objective:
# Organize the dataframe into a consistent structure.
#
# Final column order:
#
# Date | ticker | Open | High | Low | Close | Volume
# ============================================================

sp500 = sp500[
    [
        "Date",
        "ticker",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
]

sp500.head()
sp500.tail()

,Date,ticker,Open,High,Low,Close,Volume
6607,2026-04-13,^GSPC,6806.470215,6887.000000,6790.020020,6886.240234,4785840000
6608,2026-04-14,^GSPC,6910.200195,6969.419922,6905.169922,6967.379883,5032380000
6609,2026-04-15,^GSPC,6978.169922,7026.240234,6967.129883,7022.950195,5278610000
6610,2026-04-16,^GSPC,7037.779785,7051.229980,7008.520020,7041.279785,5173650000
6611,2026-04-17,^GSPC,7074.549805,7147.520020,7074.549805,7126.060059,6145300000


In [9]:
# ============================================================
# CELL 9 — DATASET INSPECTION
# ------------------------------------------------------------
# Objective:
# Perform basic sanity checks before saving the dataset.
#
# This includes:
#
# - dataset dimensions
# - first observations
# - last observations
# - time span covered by the dataset
# ============================================================

print("Dataset shape:", sp500.shape)

print("\nFirst observations:")
display(sp500.head())

print("\nLast observations:")
display(sp500.tail())

print("\nTime span of the dataset:")
print("Start date:", sp500["Date"].min())
print("End date:", sp500["Date"].max())

Dataset shape: (6612, 7)

First observations:


,Date,ticker,Open,High,Low,Close,Volume
0,2000-01-03,^GSPC,1469.250000,1478.000000,1438.359985,1455.219971,931800000
1,2000-01-04,^GSPC,1455.219971,1455.219971,1397.430054,1399.420044,1009000000
2,2000-01-05,^GSPC,1399.420044,1413.270020,1377.680054,1402.109985,1085500000
3,2000-01-06,^GSPC,1402.109985,1411.900024,1392.099976,1403.449951,1092300000
4,2000-01-07,^GSPC,1403.449951,1441.469971,1400.729980,1441.469971,1225200000



Last observations:


,Date,ticker,Open,High,Low,Close,Volume
6607,2026-04-13,^GSPC,6806.470215,6887.000000,6790.020020,6886.240234,4785840000
6608,2026-04-14,^GSPC,6910.200195,6969.419922,6905.169922,6967.379883,5032380000
6609,2026-04-15,^GSPC,6978.169922,7026.240234,6967.129883,7022.950195,5278610000
6610,2026-04-16,^GSPC,7037.779785,7051.229980,7008.520020,7041.279785,5173650000
6611,2026-04-17,^GSPC,7074.549805,7147.520020,7074.549805,7126.060059,6145300000



Time span of the dataset:
Start date: 2000-01-03 00:00:00
End date: 2026-04-17 00:00:00


In [ ]:
# ============================================================
# CELL 10 — SAVE RAW DATASET
# ------------------------------------------------------------
# Objective:
# Store the raw market dataset locally.
#
# Saving the dataset ensures:
#
# - experiment reproducibility
# - independence from external APIs
# - faster execution of subsequent notebooks
# ============================================================

file_path = raw_dir / "raw_sp500_data.csv"

sp500.to_csv(file_path, index=False)

print("Raw dataset saved to:")
print(file_path)

Raw dataset saved to:
c:\Users\anton\0______MASTER_IN_DATA_SCIENCE\1_SEMESTRE\RESEARCH_METHODOLOGIES\RESEARCH_METHODOLOGIES_2\ML_Stocks_LSTM\datasets\raw\raw_sp500_data.csv


: 